<a href="https://colab.research.google.com/github/VenkatNarayananManjunath/NN/blob/main/Exp5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# INSTALL REQUIRED LIBRARIES
!pip install face_recognition opencv-python scikit-learn pandas

# IMPORT LIBRARIES
import face_recognition
import cv2
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_lfw_people
from datetime import datetime
from google.colab.patches import cv2_imshow

# ==============================
# LOAD OPEN-SOURCE LFW DATASET
# ==============================
print("Loading LFW dataset...")
lfw = fetch_lfw_people(min_faces_per_person=5)

images = lfw.images
names = lfw.target_names
labels = lfw.target

# ==============================
# CREATE KNOWN FACE ENCODINGS
# ==============================
known_face_encodings = []
known_face_names = []

print("Encoding faces...")
for i in range(len(images)):
    # Convert grayscale to RGB
    image = np.stack((images[i],)*3, axis=-1).astype(np.uint8)
    encodings = face_recognition.face_encodings(image)

    if encodings:
        known_face_encodings.append(encodings[0])
        known_face_names.append(names[labels[i]])

print("Total known faces:", len(known_face_encodings))

# ==============================
# ATTENDANCE DICTIONARY
# ==============================
attendance = {}

def mark_attendance(name):
    time_now = datetime.now().strftime("%H:%M:%S")
    if name not in attendance:
        attendance[name] = time_now

# ==============================
# TEST IMAGE RECOGNITION
# ==============================
test_image_path = input("Enter test image path: ")
test_image = face_recognition.load_image_file(test_image_path)

face_locations = face_recognition.face_locations(test_image)
face_encodings = face_recognition.face_encodings(test_image, face_locations)

image_bgr = cv2.cvtColor(test_image, cv2.COLOR_RGB2BGR)

for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
    matches = face_recognition.compare_faces(
        known_face_encodings, face_encoding, tolerance=0.5
    )
    name = "Unknown"

    if True in matches:
        first_match_index = matches.index(True)
        name = known_face_names[first_match_index]
        mark_attendance(name)

    cv2.rectangle(image_bgr, (left, top), (right, bottom), (0, 255, 0), 2)
    cv2.putText(image_bgr, name, (left, top - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

# DISPLAY IMAGE
cv2_imshow(image_bgr)

# ==============================
# SAVE ATTENDANCE TO CSV
# ==============================
df = pd.DataFrame(attendance.items(), columns=["Name", "Time"])
date_today = datetime.now().strftime("%Y-%m-%d")
file_name = f"attendance_{date_today}.csv"
df.to_csv(file_name, index=False)

print("\nAttendance saved as:", file_name)
print(df)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for face-recognition-models: filename=face_recognition_models-0.3.0-py2.py3-none-any.whl size=100566166 sha256=42661c312678ad059fabaa150fde896afe7d1246135a76936450be2a4ae6de63
  Stored in directory: /root/.cache/pip/wheels/8f/47/c8/f44c5aebb7507f7c8a2c0bd23151d732d0f0bd6884ad4ac635
Successfully built face-recognition-models
Loading LFW dataset...
Encoding faces...
